# 温度駆動力を与える。固相液相の**多結晶**シミュレーションー＞GPU計算
# ー＞ https://doi.org/10.1016/j.mtla.2023.101702 の再現
# 固液界面のみ異方性ー＞ active parameter trackingで高速化
# ver6の三次元化

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda, float32, int32
from scipy.spatial.transform import Rotation

# =========================
# 0) Settings / Parameters
# =========================
nx, ny, nz = 256, 256, 32
number_of_grain = 17     # 0: liquid, 1..N-1: solid grains

dx, dy, dz = 1e-4, 1e-4, 1e-4
dt = 0.0001
nsteps = 10001
pi = np.pi

delta = 6.0 * dx
T_melt = 1687
G = 1.0e+02
V_pulling = 1.0e-2
Sf = 2.12e+04

# ---- Energetic anisotropy ----
a0 = 54.7 * pi / 180.0
delta_a = 0.36
mu_a = 0.6156
p_round = 0.05
ksi = 0.30
omg = 10 * pi / 180.0

# ---- Interface energies ----
gamma_100 = 0.44
gamma_GB = 0.60

# ---- Mobilities ----
M_SL = 3.0e-5
M_GB = M_SL

# Output
out_dir = "result/ver6/3d_nz32/high_only_phase"
os.makedirs(out_dir, exist_ok=True)

if number_of_grain > 20:
    raise ValueError("number_of_grain must be <= 20 because mf/KMAX are fixed to 20.")


In [ ]:
# =====================================
# 1) Grain orientations (quaternions) [3D]
#   - Liquid is dummy identity
#   - Solids are random 3D orientations
# =====================================
np.random.seed(42)
N = number_of_grain

grain_quaternions = np.zeros((N, 4), dtype=np.float64)
grain_quaternions[0] = np.array([0.0, 0.0, 0.0, 1.0])  # liquid dummy (x,y,z,w)

rng = np.random.default_rng(42)
for gid in range(1, N):
    grain_quaternions[gid] = Rotation.random(random_state=rng).as_quat()

# ---- Precompute rotated {111} normals (8) for each grain ----
n111_base = np.array([
    [ 1,  1,  1], [ 1,  1, -1], [ 1, -1,  1], [-1,  1,  1],
    [ 1, -1, -1], [-1,  1, -1], [-1, -1,  1], [-1, -1, -1],
], dtype=np.float32) / np.sqrt(3.0)

grain_n111 = np.zeros((N, 8, 3), dtype=np.float32)
for gid in range(N):
    Rmat = Rotation.from_quat(grain_quaternions[gid]).as_matrix().astype(np.float32)
    grain_n111[gid] = (Rmat @ n111_base.T).T  # (8,3)


In [ ]:
mf = np.zeros((20, nx, ny, nz), dtype=np.int32)
nf = np.zeros((nx, ny, nz), dtype=np.int32)

# ==========================================================
# 2) Constant GB interactions (wij,aij,mij)
# ==========================================================
wij = np.zeros((N, N), dtype=np.float32)
aij = np.zeros((N, N), dtype=np.float32)
mij = np.zeros((N, N), dtype=np.float32)


In [ ]:
# --- helper conversions ---
def eps_from_gamma(gamma):
    # epsilon = sqrt(8*delta*gamma/pi^2)
    return math.sqrt(8.0 * delta * gamma / (pi * pi))

def w_from_gamma(gamma):
    # w = 4*gamma/delta
    return 4.0 * gamma / delta

def mij_from_M(M):
    # mij(phi) = (pi^2/(8*delta)) * M
    return (pi * pi / (8.0 * delta)) * M

# ---- GB constants (isotropic) ----
eps_GB = eps_from_gamma(gamma_GB)
w_GB = w_from_gamma(gamma_GB)
m_GB_phi = mij_from_M(M_GB)

for i in range(1, N):
    for j in range(1, N):
        if i == j:
            continue
        wij[i, j] = w_GB
        aij[i, j] = eps_GB
        mij[i, j] = m_GB_phi

# ---- SL baseline ----
eps0_sl = eps_from_gamma(gamma_100)
w0_sl = w_from_gamma(gamma_100)
m_sl_phi = mij_from_M(M_SL)

for i in range(1, N):
    wij[0, i] = wij[i, 0] = w0_sl
    aij[0, i] = aij[i, 0] = eps0_sl
    mij[0, i] = mij[i, 0] = m_sl_phi


In [ ]:
NG = 20

@cuda.jit(device=True, inline=True)
def calc_a_from_cos(cost, a0, delta_a, mu_a, p_round):
    # Appendix A (A2)-(A4)
    c2 = cost * cost
    C = math.sqrt(c2 + p_round * p_round)
    S = math.sqrt(max(1.0 - c2, 0.0) + p_round * p_round)
    return mu_a * (1.0 + delta_a * (C + math.tan(a0) * S))

@cuda.jit(device=True, inline=True)
def calc_b_from_cos(best_cost, ksi, omg):
    # eq(7),(9)
    theta_mod = math.acos(best_cost)
    theta_omg = theta_mod * omg
    tanv = math.tan(theta_omg)
    if abs(tanv) < 1e-8:
        return ksi
    return ksi + (1.0 - ksi) * tanv * math.tanh(1.0 / tanv)


In [ ]:
@cuda.jit(device=True, inline=True)
def best_cos_from_grad(gx, gy, gz, n111, solidid, g2_floor):
    g2 = gx * gx + gy * gy + gz * gz
    if g2 < g2_floor:
        return 0.0

    inv_g = 1.0 / math.sqrt(g2)
    nxn = gx * inv_g
    nyn = gy * inv_g
    nzn = gz * inv_g

    best = 0.0
    for t in range(8):
        c = abs(
            nxn * n111[solidid, t, 0]
            + nyn * n111[solidid, t, 1]
            + nzn * n111[solidid, t, 2]
        )
        if c > best:
            best = c

    if best > 1.0:
        best = 1.0
    return best


In [ ]:
from numba import cuda, float32
import math

# -----------------------------------------------------
# helper: x periodic, y clamp, z periodic
# -----------------------------------------------------
@cuda.jit(device=True, inline=True)
def idx_xp(l, nx):
    return l + 1 if l < nx - 1 else 0

@cuda.jit(device=True, inline=True)
def idx_xm(l, nx):
    return l - 1 if l > 0 else nx - 1

@cuda.jit(device=True, inline=True)
def idx_yp(m, ny):
    return m + 1 if m < ny - 1 else ny - 1

@cuda.jit(device=True, inline=True)
def idx_ym(m, ny):
    return m - 1 if m > 0 else 0

@cuda.jit(device=True, inline=True)
def idx_zp(n, nz):
    return n + 1 if n < nz - 1 else 0

@cuda.jit(device=True, inline=True)
def idx_zm(n, nz):
    return n - 1 if n > 0 else nz - 1

@cuda.jit(device=True, inline=True)
def grad_phi_xyz(phi, gid, l, m, n, nx, ny, nz, dx, dy, dz):
    lp = idx_xp(l, nx)
    lm = idx_xm(l, nx)
    mp = idx_yp(m, ny)
    mm = idx_ym(m, ny)
    np1 = idx_zp(n, nz)
    nm1 = idx_zm(n, nz)

    gx = (phi[gid, lp, m, n] - phi[gid, lm, m, n]) / (2.0 * dx)
    gy = (phi[gid, l, mp, n] - phi[gid, l, mm, n]) / (2.0 * dy)
    gz = (phi[gid, l, m, np1] - phi[gid, l, m, nm1]) / (2.0 * dz)
    return gx, gy, gz


In [ ]:
@cuda.jit(device=True, inline=True)
def eps2_at_cell_from_liquid(
    phi, l, m, n, nx, ny, nz, dx, dy, dz,
    solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor
):
    # liquid phase gradient
    gx, gy, gz = grad_phi_xyz(phi, 0, l, m, n, nx, ny, nz, dx, dy, dz)

    # best cos(theta)
    best_cos = best_cos_from_grad(gx, gy, gz, n111, solidid, g2_floor)

    # anisotropic epsilon
    a_sl = calc_a_from_cos(best_cos, a0, delta_a, mu_a, p_round)
    eps_sl = eps0_sl * a_sl
    return eps_sl * eps_sl


In [ ]:
@cuda.jit(device=True, inline=True)
def aniso_term1_solid(
    phi, l, m, n, nx, ny, nz, dx, dy, dz,
    solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor
):
    lp = idx_xp(l, nx)
    lm = idx_xm(l, nx)
    mp = idx_yp(m, ny)
    mm = idx_ym(m, ny)
    np1 = idx_zp(n, nz)
    nm1 = idx_zm(n, nz)

    eps2_c = eps2_at_cell_from_liquid(phi, l, m, n, nx, ny, nz, dx, dy, dz, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)
    eps2_xp = eps2_at_cell_from_liquid(phi, lp, m, n, nx, ny, nz, dx, dy, dz, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)
    eps2_xm = eps2_at_cell_from_liquid(phi, lm, m, n, nx, ny, nz, dx, dy, dz, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)
    eps2_yp = eps2_at_cell_from_liquid(phi, l, mp, n, nx, ny, nz, dx, dy, dz, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)
    eps2_ym = eps2_at_cell_from_liquid(phi, l, mm, n, nx, ny, nz, dx, dy, dz, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)
    eps2_zp = eps2_at_cell_from_liquid(phi, l, m, np1, nx, ny, nz, dx, dy, dz, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)
    eps2_zm = eps2_at_cell_from_liquid(phi, l, m, nm1, nx, ny, nz, dx, dy, dz, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)

    phi_c = phi[0, l, m, n]
    phi_xp = phi[0, lp, m, n]
    phi_xm = phi[0, lm, m, n]
    phi_yp = phi[0, l, mp, n]
    phi_ym = phi[0, l, mm, n]
    phi_zp = phi[0, l, m, np1]
    phi_zm = phi[0, l, m, nm1]

    Fx_p = (eps2_c + eps2_xp) * (phi_xp - phi_c) / (2.0 * dx)
    Fx_m = (eps2_c + eps2_xm) * (phi_c - phi_xm) / (2.0 * dx)
    Fy_p = (eps2_c + eps2_yp) * (phi_yp - phi_c) / (2.0 * dy)
    Fy_m = (eps2_c + eps2_ym) * (phi_c - phi_ym) / (2.0 * dy)
    Fz_p = (eps2_c + eps2_zp) * (phi_zp - phi_c) / (2.0 * dz)
    Fz_m = (eps2_c + eps2_zm) * (phi_c - phi_zm) / (2.0 * dz)

    return (Fx_p - Fx_m) / dx + (Fy_p - Fy_m) / dy + (Fz_p - Fz_m) / dz


In [ ]:
from numba import cuda, float32
import math

# -----------------------------------------
# 2nd derivatives of phi (for liquid: gid=0)
# -----------------------------------------
@cuda.jit(device=True, inline=True)
def d2_phi_xyz(phi, gid, l, m, n, nx, ny, nz, dx, dy, dz):
    lp = idx_xp(l, nx)
    lm = idx_xm(l, nx)
    mp = idx_yp(m, ny)
    mm = idx_ym(m, ny)
    np1 = idx_zp(n, nz)
    nm1 = idx_zm(n, nz)

    c = phi[gid, l, m, n]
    xp = phi[gid, lp, m, n]
    xm = phi[gid, lm, m, n]
    yp = phi[gid, l, mp, n]
    ym = phi[gid, l, mm, n]
    zp = phi[gid, l, m, np1]
    zm = phi[gid, l, m, nm1]

    phixx = (xp - 2.0 * c + xm) / (dx * dx)
    phiyy = (yp - 2.0 * c + ym) / (dy * dy)
    phizz = (zp - 2.0 * c + zm) / (dz * dz)

    xpy = phi[gid, lp, mp, n]
    xpm = phi[gid, lp, mm, n]
    xmy = phi[gid, lm, mp, n]
    xmm = phi[gid, lm, mm, n]
    phixy = (xpy - xpm - xmy + xmm) * (0.25 / (dx * dy))

    xpz = phi[gid, lp, m, np1]
    xpn = phi[gid, lp, m, nm1]
    xmz = phi[gid, lm, m, np1]
    xmn = phi[gid, lm, m, nm1]
    phixz = (xpz - xpn - xmz + xmn) * (0.25 / (dx * dz))

    ypz = phi[gid, l, mp, np1]
    ypn = phi[gid, l, mp, nm1]
    ymz = phi[gid, l, mm, np1]
    ymn = phi[gid, l, mm, nm1]
    phiyz = (ypz - ypn - ymz + ymn) * (0.25 / (dy * dz))

    return phixx, phiyy, phizz, phixy, phixz, phiyz


# -------------------------------------------------------
# Facet selection: return signed cos, abs cos, and (nx,ny,nz)
# -------------------------------------------------------
@cuda.jit(device=True, inline=True)
def facet_cos_and_nxyz_from_grad(phix, phiy, phiz, n111, solidid, g2_floor):
    q = phix * phix + phiy * phiy + phiz * phiz
    if q < g2_floor:
        return float32(0.0), float32(0.0), float32(1.0), float32(0.0), float32(0.0)

    inv_g = 1.0 / math.sqrt(q)
    gx = phix * inv_g
    gy = phiy * inv_g
    gz = phiz * inv_g

    best_abs = 0.0
    best_signed = 0.0
    best_nx = 1.0
    best_ny = 0.0
    best_nz = 0.0

    for t in range(8):
        ux = n111[solidid, t, 0]
        uy = n111[solidid, t, 1]
        uz = n111[solidid, t, 2]
        dot = gx * ux + gy * uy + gz * uz
        ad = abs(dot)
        if ad > best_abs:
            best_abs = ad
            best_signed = dot
            best_nx = ux
            best_ny = uy
            best_nz = uz

    if best_abs > 1.0:
        best_abs = 1.0
    if best_signed > 1.0:
        best_signed = 1.0
    if best_signed < -1.0:
        best_signed = -1.0

    return float32(best_signed), float32(best_abs), float32(best_nx), float32(best_ny), float32(best_nz)


# -------------------------------------------------------
# A12 + A14/A15 : return (a, da/dphix, da/dphiy, da/dphiz)
# -------------------------------------------------------
@cuda.jit(device=True, inline=True)
def da_dphixyz_A12(phi, l, m, n, nx, ny, nz, dx, dy, dz,
                   solidid,
                   a0, delta_a, mu_a, p_round,
                   n111, g2_floor):

    phix, phiy, phiz = grad_phi_xyz(phi, 0, l, m, n, nx, ny, nz, dx, dy, dz)
    q = phix * phix + phiy * phiy + phiz * phiz
    if q < g2_floor:
        return float32(0.0), float32(0.0), float32(0.0), float32(0.0)

    cos_signed, cos_abs, nxn, nyn, nzn = facet_cos_and_nxyz_from_grad(phix, phiy, phiz, n111, solidid, g2_floor)
    a_val = calc_a_from_cos(cos_abs, a0, delta_a, mu_a, p_round)

    g = math.sqrt(q)
    inv_g = 1.0 / g
    inv_g3 = 1.0 / (q * g)

    udotg = phix * nxn + phiy * nyn + phiz * nzn
    dc_dphix = nxn * inv_g - udotg * phix * inv_g3
    dc_dphiy = nyn * inv_g - udotg * phiy * inv_g3
    dc_dphiz = nzn * inv_g - udotg * phiz * inv_g3

    c = cos_signed
    C = math.sqrt(c * c + p_round * p_round)
    S = math.sqrt(max(1.0 - c * c, 0.0) + p_round * p_round)
    coef = (c / C) - math.tan(a0) * (c / S)

    da_dphix = mu_a * delta_a * coef * dc_dphix
    da_dphiy = mu_a * delta_a * coef * dc_dphiy
    da_dphiz = mu_a * delta_a * coef * dc_dphiz

    return float32(a_val), float32(da_dphix), float32(da_dphiy), float32(da_dphiz)


# -------------------------------------------------------
# A13 : d/dx(da/dphix), d/dy(da/dphiy), d/dz(da/dphiz)
# -------------------------------------------------------
@cuda.jit(device=True, inline=True)
def d_dxyz_da_dphip_A13(phi, l, m, n, nx, ny, nz, dx, dy, dz,
                        solidid,
                        a0, delta_a, mu_a, p_round,
                        n111, g2_floor):

    lp = idx_xp(l, nx)
    lm = idx_xm(l, nx)
    mp = idx_yp(m, ny)
    mm = idx_ym(m, ny)
    np1 = idx_zp(n, nz)
    nm1 = idx_zm(n, nz)

    _, da_dphix_lp, _, _ = da_dphixyz_A12(phi, lp, m, n, nx, ny, nz, dx, dy, dz, solidid, a0, delta_a, mu_a, p_round, n111, g2_floor)
    _, da_dphix_lm, _, _ = da_dphixyz_A12(phi, lm, m, n, nx, ny, nz, dx, dy, dz, solidid, a0, delta_a, mu_a, p_round, n111, g2_floor)

    _, _, da_dphiy_mp, _ = da_dphixyz_A12(phi, l, mp, n, nx, ny, nz, dx, dy, dz, solidid, a0, delta_a, mu_a, p_round, n111, g2_floor)
    _, _, da_dphiy_mm, _ = da_dphixyz_A12(phi, l, mm, n, nx, ny, nz, dx, dy, dz, solidid, a0, delta_a, mu_a, p_round, n111, g2_floor)

    _, _, _, da_dphiz_np = da_dphixyz_A12(phi, l, m, np1, nx, ny, nz, dx, dy, dz, solidid, a0, delta_a, mu_a, p_round, n111, g2_floor)
    _, _, _, da_dphiz_nm = da_dphixyz_A12(phi, l, m, nm1, nx, ny, nz, dx, dy, dz, solidid, a0, delta_a, mu_a, p_round, n111, g2_floor)

    d_dx = (da_dphix_lp - da_dphix_lm) * (0.5 / dx)
    d_dy = (da_dphiy_mp - da_dphiy_mm) * (0.5 / dy)
    d_dz = (da_dphiz_np - da_dphiz_nm) * (0.5 / dz)

    return float32(d_dx), float32(d_dy), float32(d_dz)


# -------------------------------------------------------
# Torque term A11 (3D)
# -------------------------------------------------------
@cuda.jit(device=True, inline=True)
def torque_A11(phi, l, m, n, nx, ny, nz, dx, dy, dz, solidid, eps0_sl,
               a0, delta_a, mu_a, p_round,
               n111, g2_floor):

    phix, phiy, phiz = grad_phi_xyz(phi, 0, l, m, n, nx, ny, nz, dx, dy, dz)
    q = phix * phix + phiy * phiy + phiz * phiz
    if q < g2_floor:
        return 0.0

    phixx, phiyy, phizz, phixy, phixz, phiyz = d2_phi_xyz(phi, 0, l, m, n, nx, ny, nz, dx, dy, dz)

    q_x = 2.0 * (phix * phixx + phiy * phixy + phiz * phixz)
    q_y = 2.0 * (phix * phixy + phiy * phiyy + phiz * phiyz)
    q_z = 2.0 * (phix * phixz + phiy * phiyz + phiz * phizz)

    a_val, da_dphix, da_dphiy, da_dphiz = da_dphixyz_A12(
        phi, l, m, n, nx, ny, nz, dx, dy, dz,
        solidid, a0, delta_a, mu_a, p_round,
        n111, g2_floor
    )

    cos_signed, _, nxn, nyn, nzn = facet_cos_and_nxyz_from_grad(phix, phiy, phiz, n111, solidid, g2_floor)

    g = math.sqrt(q)
    inv_g = 1.0 / g
    inv_g3 = 1.0 / (q * g)
    udotg = phix * nxn + phiy * nyn + phiz * nzn

    dc_dphix = nxn * inv_g - udotg * phix * inv_g3
    dc_dphiy = nyn * inv_g - udotg * phiy * inv_g3
    dc_dphiz = nzn * inv_g - udotg * phiz * inv_g3

    c_x = dc_dphix * phixx + dc_dphiy * phixy + dc_dphiz * phixz
    c_y = dc_dphix * phixy + dc_dphiy * phiyy + dc_dphiz * phiyz
    c_z = dc_dphix * phixz + dc_dphiy * phiyz + dc_dphiz * phizz

    c = cos_signed
    C = math.sqrt(c * c + p_round * p_round)
    S = math.sqrt(max(1.0 - c * c, 0.0) + p_round * p_round)
    coef = (c / C) - math.tan(a0) * (c / S)

    da_dx = mu_a * delta_a * coef * c_x
    da_dy = mu_a * delta_a * coef * c_y
    da_dz = mu_a * delta_a * coef * c_z

    d_dx_da_dphix, d_dy_da_dphiy, d_dz_da_dphiz = d_dxyz_da_dphip_A13(
        phi, l, m, n, nx, ny, nz, dx, dy, dz,
        solidid, a0, delta_a, mu_a, p_round,
        n111, g2_floor
    )

    Ex = (da_dx * da_dphix * q) + (a_val * d_dx_da_dphix * q) + (a_val * da_dphix * q_x)
    Ey = (da_dy * da_dphiy * q) + (a_val * d_dy_da_dphiy * q) + (a_val * da_dphiy * q_y)
    Ez = (da_dz * da_dphiz * q) + (a_val * d_dz_da_dphiz * q) + (a_val * da_dphiz * q_z)

    return (eps0_sl * eps0_sl) * (Ex + Ey + Ez)


In [ ]:
@cuda.jit
def kernel_update_nfmf(phi, mf, nf, nx, ny, nz, number_of_grain):
    l, m, n = cuda.grid(3)
    if l >= nx or m >= ny or n >= nz:
        return

    l_p = idx_xp(l, nx)
    l_m = idx_xm(l, nx)
    m_p = idx_yp(m, ny)
    m_m = idx_ym(m, ny)
    n_p = idx_zp(n, nz)
    n_m = idx_zm(n, nz)

    count = 0
    for i in range(number_of_grain):
        if (phi[i, l, m, n] > 0.0) or ((phi[i, l, m, n] == 0.0) and (
            (phi[i, l_p, m, n] > 0.0) or (phi[i, l_m, m, n] > 0.0) or
            (phi[i, l, m_p, n] > 0.0) or (phi[i, l, m_m, n] > 0.0) or
            (phi[i, l, m, n_p] > 0.0) or (phi[i, l, m, n_m] > 0.0)
        )):
            count += 1
            mf[count - 1, l, m, n] = i

    nf[l, m, n] = count


In [ ]:
LIQ = 0
KMAX = 20

@cuda.jit
def kernel_update_phasefield_active(phi, phi_new, temp, mf, nf,
                                    wij, aij, mij, n111,
                                    nx, ny, nz, number_of_grain,
                                    dx, dy, dz, dt, T_melt, Sf,
                                    eps0_sl, w0_sl,
                                    a0, delta_a, mu_a, p_round,
                                    g2_floor, ksi, omg):

    l, m, n = cuda.grid(3)
    if l >= nx or m >= ny or n >= nz:
        return

    lp = idx_xp(l, nx)
    lm = idx_xm(l, nx)
    mp = idx_yp(m, ny)
    mm = idx_ym(m, ny)
    np1 = idx_zp(n, nz)
    nm1 = idx_zm(n, nz)

    inv_dx2 = 1.0 / (dx * dx)
    inv_dy2 = 1.0 / (dy * dy)
    inv_dz2 = 1.0 / (dz * dz)

    Tcur = temp[l, m, n]
    nact = nf[l, m, n]

    # fallback for unexpected empty-active cell
    if nact <= 0:
        for i in range(number_of_grain):
            phi_new[i, l, m, n] = 0.0
        phi_new[LIQ, l, m, n] = 1.0
        return

    lap_sl = cuda.local.array(KMAX, float32)
    b_sl = cuda.local.array(KMAX, float32)

    for t in range(nact):
        gid = mf[t, l, m, n]
        if gid == LIQ:
            b_sl[t] = 1.0
            lap_sl[t] = 0.0
        else:
            lap_sl[t] = aniso_term1_solid(
                phi, l, m, n, nx, ny, nz, dx, dy, dz,
                gid, eps0_sl,
                a0, delta_a, mu_a, p_round,
                n111, g2_floor
            )
            lap_sl[t] += torque_A11(
                phi, l, m, n, nx, ny, nz, dx, dy, dz,
                gid, eps0_sl,
                a0, delta_a, mu_a, p_round,
                n111, g2_floor
            )

            gx, gy, gz = grad_phi_xyz(phi, LIQ, l, m, n, nx, ny, nz, dx, dy, dz)
            best_cos = best_cos_from_grad(gx, gy, gz, n111, gid, g2_floor)
            b_sl[t] = calc_b_from_cos(best_cos, ksi, omg)

    # choose dominant solid phase to evaluate w_sl
    i_s = 1
    maxv = 0.0
    for t in range(nact):
        gid = mf[t, l, m, n]
        if gid == LIQ:
            continue
        v = phi[gid, l, m, n]
        if v > maxv:
            maxv = v
            i_s = gid

    phix = (phi[i_s, lp, m, n] - phi[i_s, lm, m, n]) * (0.5 / dx)
    phiy = (phi[i_s, l, mp, n] - phi[i_s, l, mm, n]) * (0.5 / dy)
    phiz = (phi[i_s, l, m, np1] - phi[i_s, l, m, nm1]) * (0.5 / dz)
    gn2 = phix * phix + phiy * phiy + phiz * phiz

    cmax = 0.0
    if gn2 >= g2_floor:
        inv_gn = 1.0 / math.sqrt(gn2)
        nx_ = phix * inv_gn
        ny_ = phiy * inv_gn
        nz_ = phiz * inv_gn
        for tt in range(8):
            c = abs(nx_ * n111[i_s, tt, 0] + ny_ * n111[i_s, tt, 1] + nz_ * n111[i_s, tt, 2])
            if c > cmax:
                cmax = c
        if cmax > 1.0:
            cmax = 1.0

    a_loc = calc_a_from_cos(cmax, a0, delta_a, mu_a, p_round)
    w_sl = w0_sl * (a_loc * a_loc)

    for i in range(number_of_grain):
        phi_new[i, l, m, n] = 0.0

    for t1 in range(nact):
        i = mf[t1, l, m, n]
        dpi = 0.0

        for t2 in range(nact):
            j = mf[t2, l, m, n]
            if i == j:
                continue

            driving_force = 0.0
            if (i != LIQ and j == LIQ):
                driving_force = -Sf * (Tcur - T_melt)
            elif (i == LIQ and j != LIQ):
                driving_force = Sf * (Tcur - T_melt)

            ppp = 0.0
            for t3 in range(nact):
                k = mf[t3, l, m, n]
                cphi = phi[k, l, m, n]

                lap_k = (
                    (phi[k, lp, m, n] - 2.0 * cphi + phi[k, lm, m, n]) * inv_dx2
                    + (phi[k, l, mp, n] - 2.0 * cphi + phi[k, l, mm, n]) * inv_dy2
                    + (phi[k, l, m, np1] - 2.0 * cphi + phi[k, l, m, nm1]) * inv_dz2
                )

                wik = wij[i, k]
                wjk = wij[j, k]
                if k == LIQ:
                    if i != LIQ:
                        wik = w_sl
                    if j != LIQ:
                        wjk = w_sl
                term1 = (wik - wjk) * cphi

                term2 = 0.0
                if (j == LIQ and i != LIQ):
                    if k == i:
                        term2 = 0.5 * lap_sl[t1]
                elif (i == LIQ and j != LIQ):
                    if k == j:
                        term2 = -0.5 * lap_sl[t2]
                else:
                    eps2ik = aij[i, k] * aij[i, k]
                    eps2jk = aij[j, k] * aij[j, k]
                    term2 = 0.5 * (eps2ik - eps2jk) * lap_k

                ppp += term1 + term2

            phii_phij = phi[i, l, m, n] * phi[j, l, m, n]
            term_force = (8.0 / 3.1415926535) * math.sqrt(max(phii_phij, 0.0)) * driving_force

            mij_eff = mij[i, j]
            if i != LIQ and j == LIQ:
                mij_eff = mij[i, j] * b_sl[t1]
            elif i == LIQ and j != LIQ:
                mij_eff = mij[i, j] * b_sl[t2]

            dpi -= 2.0 * mij_eff / float(nact) * (ppp - term_force)

        phi_new[i, l, m, n] = phi[i, l, m, n] + dt * dpi

    # projection / normalization on active phases
    s = 0.0
    for t in range(nact):
        i = mf[t, l, m, n]
        v = phi_new[i, l, m, n]

        if v < 0.0:
            v = 0.0
        if v > 1.0:
            v = 1.0

        phi_new[i, l, m, n] = v
        s += v

    if s > 1e-20:
        invs = 1.0 / s
        for t in range(nact):
            i = mf[t, l, m, n]
            phi_new[i, l, m, n] *= invs
    else:
        phi_new[LIQ, l, m, n] = 1.0


In [ ]:
@cuda.jit
def kernel_update_temp(temp, cooling_rate, nx, ny, nz):
    l, m, n = cuda.grid(3)
    if l < nx and m < ny and n < nz:
        temp[l, m, n] -= cooling_rate


In [ ]:
# =====================================================
# 4) Initialization (3D)
# =====================================================
phi_cpu = np.zeros((number_of_grain, nx, ny, nz), dtype=np.float32)
seed_height = 32
factor = 2.2 / delta

n_solid = number_of_grain - 1
grain_width = max(1, nx // n_solid)

for l in range(nx):
    grain_id = int(l // grain_width) + 1
    if grain_id > n_solid:
        grain_id = n_solid
    for m in range(ny):
        y = m * dy
        dist = y - (seed_height * dy)
        phi_solid = 0.5 * (1.0 - np.tanh(factor * dist))
        for n in range(nz):
            phi_cpu[grain_id, l, m, n] = phi_solid
            phi_cpu[0, l, m, n] = 1.0 - phi_solid

temp_cpu = np.zeros((nx, ny, nz), dtype=np.float64)
for m in range(ny):
    temp_cpu[:, m, :] = T_melt + G * (m - seed_height) * dy

mid_z = nz // 2
phase_map = np.argmax(phi_cpu[:, :, :, mid_z], axis=0)
temp_slice = temp_cpu[:, :, mid_z]

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
ax1, ax2 = axes

im1 = ax1.imshow(phase_map.T, cmap="tab20", origin="lower")
cbar1 = fig.colorbar(im1, ax=ax1, ticks=np.arange(number_of_grain), shrink=0.7)
cbar1.set_label("Phase ID", fontsize=30)
cbar1.ax.tick_params(labelsize=20)
ax1.set_title(f"Initial Grain Map (z={mid_z})", fontsize=30)
ax1.axis("off")

im2 = ax2.imshow(temp_slice.T, cmap="hot", origin="lower")
cbar2 = fig.colorbar(im2, ax=ax2, shrink=0.7)
cbar2.set_label("Temperature [K]", fontsize=30)
cbar2.ax.tick_params(labelsize=20)
ax2.set_title(f"Initial Temperature (z={mid_z})", fontsize=30)
ax2.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(out_dir, "initial_grain_temp.png"), dpi=150)


In [ ]:
# dtype
phi_cpu = phi_cpu.astype(np.float32)
temp_cpu = temp_cpu.astype(np.float32)
mf_cpu = mf.astype(np.int32)
nf_cpu = nf.astype(np.int32)

d_phi = cuda.to_device(phi_cpu)
d_phi_new = cuda.to_device(phi_cpu)
d_temp = cuda.to_device(temp_cpu)
d_mf = cuda.to_device(mf_cpu)
d_nf = cuda.to_device(nf_cpu)
d_wij = cuda.to_device(wij.astype(np.float32))
d_aij = cuda.to_device(aij.astype(np.float32))
d_mij = cuda.to_device(mij.astype(np.float32))
d_n111 = cuda.to_device(grain_n111.astype(np.float32))

threadsperblock = (8, 8, 4)
blockspergrid = (
    math.ceil(nx / threadsperblock[0]),
    math.ceil(ny / threadsperblock[1]),
    math.ceil(nz / threadsperblock[2]),
)

cooling_rate = np.float32(G * V_pulling * dt)
T_melt_f = np.float32(T_melt)
Sf_f = np.float32(Sf)


In [ ]:
# =====================================================
# 6) Main loop
# =====================================================
save_every = 1000
g2_floor_f = np.float32((0.1 / min(dx, dy, dz)) ** 2)
mid_z = nz // 2

for nstep in range(1, nsteps + 1):
    kernel_update_temp[blockspergrid, threadsperblock](d_temp, cooling_rate, nx, ny, nz)
    kernel_update_nfmf[blockspergrid, threadsperblock](d_phi, d_mf, d_nf, nx, ny, nz, number_of_grain)

    kernel_update_phasefield_active[blockspergrid, threadsperblock](
        d_phi, d_phi_new, d_temp, d_mf, d_nf,
        d_wij, d_aij, d_mij, d_n111,
        nx, ny, nz, number_of_grain,
        np.float32(dx), np.float32(dy), np.float32(dz), np.float32(dt),
        T_melt_f, Sf_f,
        np.float32(eps0_sl), np.float32(w0_sl),
        np.float32(a0), np.float32(delta_a), np.float32(mu_a), np.float32(p_round),
        g2_floor_f, np.float32(ksi), np.float32(omg)
    )

    d_phi, d_phi_new = d_phi_new, d_phi

    if nstep % save_every == 0:
        current_phi = d_phi.copy_to_host()
        phase_map = np.argmax(current_phi[:, :, :, mid_z], axis=0)

        fig, ax1 = plt.subplots(1, 1, figsize=(10, 10))
        im1 = ax1.imshow(phase_map.T, cmap="tab20", origin="lower")
        cbar1 = fig.colorbar(im1, ax=ax1, ticks=np.arange(number_of_grain), shrink=0.7)
        cbar1.set_label("Phase ID", font="Times New Roman", fontsize=32)
        cbar1.ax.tick_params(labelsize=20)

        ax1.set_title(
            f"coolingrate = {V_pulling * G} K/s\nstep {nstep} (z={mid_z})",
            font="Times New Roman",
            fontsize=34,
        )
        ax1.axis("off")

        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"step_{nstep}.png"), dpi=150)
        plt.close()